# 99 · Run the whole pipeline in one go
Use this after notebook 00 has worked interactively. Click **Save Version → Save & Run All (Commit)**. Kaggle runs it in the background (up to 12 h) and keeps everything written to `/kaggle/working` in the notebook's **Output** tab. Download `fakelens_results.zip` from there when it finishes.

In [ ]:
# --- Kaggle setup (run first in every notebook) ---
# Settings: Accelerator = GPU (T4 x2 or P100), Internet = ON.
# Add Data: vtphatt2/GenImage-stable-diffusion-v1-4 and vtphatt2/GenImage-Validation
#           (+ the output dataset of the previous notebook, if you split stages across sessions).
import os, subprocess
REPO = "https://github.com/daniyal5293/fakelens.git"   # <- change, or upload the repo as a Kaggle dataset
if not os.path.exists("/kaggle/working/fakelens"):
    subprocess.run(["git", "clone", REPO, "/kaggle/working/fakelens"], check=True)
else:
    subprocess.run(["git", "-C", "/kaggle/working/fakelens", "pull"], check=False)
os.chdir("/kaggle/working/fakelens")
!pip install -q open_clip_torch
!pip install -q -e . --no-deps
# Show the dataset folder structure (look for .../train/ai, .../train/nature, .../val/ai, .../val/nature):
!find /kaggle/input -maxdepth 7 -type d | head -80

In [ ]:
# Check the dataset folder names printed above and fix them here if they differ.
CFG = "configs/default.yaml"
SET = "--set roots.train_root=/kaggle/input/datasets/vtphatt2/genimage-stable-diffusion-v1-4 roots.test_root=/kaggle/input/datasets/vtphatt2/genimage-validation"
RUNS = "/kaggle/working/runs"

In [ ]:
# Write a Kaggle config with the dataset paths from the cell above
import yaml
cfg = yaml.safe_load(open('configs/default.yaml'))
for kv in SET.replace('--set', '').split():
    k, v = kv.split('=', 1); cfg['roots'][k.split('.')[1]] = v
yaml.safe_dump(cfg, open('configs/kaggle.yaml', 'w'))
print(cfg['roots'])

In [ ]:
!CFG=configs/kaggle.yaml bash scripts/run_all.sh 2>&1 | tee /kaggle/working/run_all.log

In [ ]:
!cd /kaggle/working/fakelens && zip -qr /kaggle/working/fakelens_results.zip results data/manifests checkpoints
!ls -lh /kaggle/working/fakelens_results.zip